# Week-1 Streaming Lab — Objectives

- Build an Eventstream → Eventhouse ingestion flow (conceptual and sample code)
- Process streaming data using Spark Structured Streaming (demo micro-batches)
- Query streaming windows with KQL (examples) and compare accelerated vs non-accelerated shortcut behavior

---

## Prerequisites

- Recommended: Fabric trial tenant with Eventstreams/Eventhouse access
- Local dev: Python 3.8+, pandas, pyspark (optional), and sample CSVs to simulate events
- See `DP-700-Study-Plan.md` for Real-Time Intelligence learning path link

In [ ]:
### Setup & Imports

```python
# Optional installation (uncomment when needed)
# !pip install pandas pyspark

import pandas as pd
import numpy as np
import pathlib

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup complete')
```

In [ ]:
# Data creation: synthetic event stream (micro-batches saved as files)

events = pd.DataFrame({
    'event_id': np.arange(1,21),
    'ts': pd.date_range('2026-01-01', periods=20, freq='T'),
    'user': np.random.choice(['alice','bob','charlie'], 20),
    'value': np.random.randint(1,100, 20)
})

path = 'labs/week1/data/stream_events.csv'
pathp = pathlib.Path(path)
pathp.parent.mkdir(parents=True, exist_ok=True)
events.to_csv(path, index=False)

events.head()


In [ ]:
# Core implementation (example Spark structured streaming snippet)

# NOTE: The following is example code for Spark structured streaming.
# When testing in Fabric, run equivalent code in a Spark notebook.

spark_snippet = '''
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Read streaming CSV files from a folder
df = spark.readStream\
    .option("header","true")\
    .csv("/path/to/stream/folder")

# Simple aggregation example
agg = df.groupBy(window(df.ts, '1 minute'), df.user).agg({'value':'sum'})

query = agg.writeStream.outputMode('complete').format('console').start()
query.awaitTermination(10)
'''

print('Spark streaming example (paste into a Spark notebook):\n')
print(spark_snippet)

# KQL example (conceptual):
kql_example = """
EventStream
| where Timestamp between (ago(5m) .. now())
| summarize count() by bin(Timestamp, 1m), user
"""
print('\nKQL example (windowing):\n', kql_example)


In [ ]:
# Tests & example run (simulate micro-batch processing with pandas)

def process_batch(df: pd.DataFrame) -> pd.DataFrame:
    # simple rolling sum by user per minute
    df['ts'] = pd.to_datetime(df['ts'])
    out = df.groupby([pd.Grouper(key='ts', freq='1T'),'user'])['value'].sum().reset_index()
    out = out.sort_values(['ts','user'])
    return out

out = process_batch(events)
print('Aggregated sample:')
print(out.head())

# Quick assertion check
assert 'value' in out.columns
print('Basic checks passed')
